# Deepfake Detection — Swin3D-only (Video Swin Transformer), 24-frame

A single-stream video deepfake detector built on a **Video Swin Transformer (Swin3D)**
(`torchvision.models.video.swin3d_t`, Kinetics-400 pretrained), fine-tuned with **LoRA** plus an
unfrozen final stage, and classified **directly by a `Linear` head**. The former ST-GCN
landmark-motion stream, the gated-fusion head, and temperature calibration have been **removed
entirely** — this is the Swin3D-only variant.

**Pipeline**

| Stage | Purpose |
|---|---|
| Setup & config | Deterministic seeds, paths, Swin3D hyper-parameters |
| MTCNN crop | The single face-crop pipeline shared by training and inference |
| Index & split | Identity-aware, leakage-safe train/val/test grouping |
| **EDA** | Class balance, per-video frame counts, resolution / FPS, sample grids, data-health checks |
| Swin3D dataset | Uniform temporal sampling of **24 frames** per clip → `(B, C, 24, H, W)` |
| Train | Real LoRA + last-stage fine-tune, class-weighted loss, cosine LR, early stopping |
| **Comparison** | Existing **32-frame pretrained** checkpoint vs the **24-frame** model trained here |

**Architecture-parity note (32 → 24 frames):** only the temporal-sampling count changes. Swin3D
depth, window structure, feature channels, the `Linear` classifier head, the loss, and the
optimizer are all identical. Swin3D pads the temporal axis to its window size internally, and the
classifier head operates on the pooled embedding (independent of `T`), so a 24-frame clip needs no
architectural change — the input tensor is simply `(B, C, 24, H, W)` instead of `(B, C, 32, H, W)`.

**Assumes:** a pre-cropped frame dataset at `FACE_ROOT` (`Real/<id>/*.jpg`, `Fake/<subcat>/<id>/*.jpg`).
Edit the config cell below before running. Training / evaluation require a GPU (e.g. Kaggle).


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | Total VRAM: {props.total_memory / 1e9:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Setup & Installs

`facenet-pytorch` provides MTCNN (the face cropper). MediaPipe / TensorFlow are **not** installed —
they were only needed by the removed ST-GCN landmark stream.


In [ ]:
!pip install -q --no-deps facenet-pytorch

import torch, numpy as np
print("torch:", torch.__version__, "| numpy:", np.__version__)

In [ ]:
import os, gc, json, random, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve,
    classification_report, balanced_accuracy_score,
)

# ------------------------------------------------------------
# Deterministic seeding for full reproducibility
# ------------------------------------------------------------
SEED = 42

def set_deterministic(seed=SEED):
    """Seed python / numpy / torch (CPU+CUDA) and force deterministic cuDNN."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_deterministic(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(f"torch {torch.__version__} | numpy {np.__version__} | seed {SEED}")


## 2. Config — EDIT THESE PATHS

Set `FACE_ROOT` (pre-cropped frames) and, optionally, `RAW_VIDEO_ROOT` (raw videos — used **only**
by the EDA stage to report true FPS / duration). The 32-frame pretrained checkpoint used by the
comparison stage is `PRETRAINED_32F_CKPT`. **`NUM_FRAMES_SWIN = 24` is the only temporal-sampling
change** versus the previous 32-frame pipeline.


In [ ]:
# ============================================================
# EDIT ME — paths
# ============================================================
FACE_ROOT = "/kaggle/input/datasets/fypset/ffpp_32frame22k"   # pre-cropped frames
REAL_DIRNAME = "Real"
FAKE_DIRNAME = "Fake"
FRAME_EXTS = (".jpg", ".jpeg", ".png")

# Optional: a raw-video source used ONLY by the EDA stage to report true FPS / duration.
# Leave as "" when you only have pre-cropped frames (EDA then reports that FPS/duration are
# not recoverable from stills and gives a nominal estimate instead of fabricating a number).
RAW_VIDEO_ROOT = ""

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MAP = {"Real": 0, "Fake": 1}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}
LABEL2ID = LABEL_MAP

# ------------------------------------------------------------
# Temporal sampling — THE ONLY CHANGE vs the 32-frame pipeline
# ------------------------------------------------------------
NUM_FRAMES_CACHED  = 30    # frames read from disk per video (unchanged)
NUM_FRAMES_SWIN    = 24    # <-- was 32: uniformly sample 24 frames per clip
SWIN_PAD_NOISE_STD = 3.0   # noise std for padded frames when fewer than k exist
IMG_SIZE = 224

# --- MTCNN crop config (identical values are saved for inference) ---
MTCNN_CROP_MODE = "pad"
MTCNN_MARGIN = 0
MTCNN_MIN_FACE_SIZE = 40
MTCNN_THRESHOLDS = [0.6, 0.7, 0.7]
MTCNN_MARGIN_JITTER = 15

# --- Swin3D appearance stream (architecture & optimization UNCHANGED) ---
SWIN_VARIANT = "swin3d_t"
SWIN_LORA_RANK = 32
SWIN_LORA_ALPHA = 64
SWIN_LORA_DROPOUT = 0.05
SWIN_TARGET_MODULES = ["qkv", "proj"]
SWIN_UNFREEZE_LAST_STAGE = True      # LoRA alone underfits -> also fine-tune final stage + norm
SWIN_BATCH_SIZE = 2
SWIN_GRAD_ACCUM = 8
SWIN_EPOCHS = 40
SWIN_PATIENCE = 12
SWIN_LR = 2e-4
SWIN_BACKBONE_LR = 2e-5
SWIN_WEIGHT_DECAY = 1e-4
SWIN_LABEL_SMOOTHING = 0.03

# --- 32-frame vs 24-frame comparison ---
RUN_COMPARISON = True
PRETRAINED_32F_CKPT = "/kaggle/input/datasets/amirali/swin-lora-final"  # existing 32-frame model
COMPARE_NUM_FRAMES_PRETRAINED = 32   # frames the pretrained checkpoint expects
COMPARE_MAX_TIMING_VIDEOS = 64       # cap videos used for the speed / VRAM probe

# --- EDA scan caps (keep EDA fast on ~22k videos) ---
EDA_MAX_VIDEOS = 1500            # videos sampled for frame-count / resolution stats
EDA_HEALTH_SCAN_LIMIT = 1500     # videos scanned for corrupt / missing frames
EDA_GRID_VIDEOS_PER_CLASS = 3    # videos shown per class in the sample-frame grid
EDA_GRID_FRAMES_PER_VIDEO = 6    # frames shown per video in the grid
EDA_NOMINAL_FPS = 30.0           # only used for the nominal duration estimate when no raw videos

VAL_SPLIT = 0.1
TEST_SPLIT = 0.1
print("config ready | NUM_FRAMES_SWIN =", NUM_FRAMES_SWIN)


## 3. MTCNN face cropping — the ONE crop pipeline used everywhere

Used to (a) crop raw videos into frames when needed and (b) written verbatim into
`predict_video.py` later, so training and inference preprocessing can never diverge.


In [ ]:
import random as _random
from facenet_pytorch import MTCNN
from PIL import Image

_mtcnn_singleton = None
def _get_mtcnn():
    global _mtcnn_singleton
    if _mtcnn_singleton is None:
        _mtcnn_singleton = MTCNN(
            image_size=IMG_SIZE,
            margin=0,                    # we apply margin ourselves in crop_face_mtcnn
            min_face_size=MTCNN_MIN_FACE_SIZE,
            thresholds=MTCNN_THRESHOLDS,
            post_process=False,
            select_largest=True,
            device=device,
            keep_all=False,
        )
    return _mtcnn_singleton


def _expand_box_pad(box, margin, w, h):
    x1, y1, x2, y2 = box
    x1, y1, x2, y2 = x1 - margin, y1 - margin, x2 + margin, y2 + margin
    bw, bh = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(bw, bh)
    return cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2


def crop_face_mtcnn(frame_bgr, mode=MTCNN_CROP_MODE, image_size=IMG_SIZE, margin=MTCNN_MARGIN):
    """Detect the largest face and return a square crop resized to image_size.
    Falls back to the resized full frame if no face is detected."""
    h, w = frame_bgr.shape[:2]
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)

    mtcnn = _get_mtcnn()
    boxes, probs = mtcnn.detect(pil_img)
    if boxes is None or len(boxes) == 0:
        return cv2.resize(frame_bgr, (image_size, image_size))

    box = boxes[0]
    x1, y1, x2, y2 = _expand_box_pad(box, margin, w, h)
    side = int(round(x2 - x1))
    if side <= 0:
        return cv2.resize(frame_bgr, (image_size, image_size))

    x1_i, y1_i = int(np.floor(x1)), int(np.floor(y1))
    x2_i, y2_i = x1_i + side, y1_i + side

    canvas = np.zeros((side, side, 3), dtype=np.uint8)
    src_x1, src_y1 = max(x1_i, 0), max(y1_i, 0)
    src_x2, src_y2 = min(x2_i, w), min(y2_i, h)
    if src_x2 <= src_x1 or src_y2 <= src_y1:
        return cv2.resize(frame_bgr, (image_size, image_size))

    dst_x1, dst_y1 = src_x1 - x1_i, src_y1 - y1_i
    dst_x2, dst_y2 = dst_x1 + (src_x2 - src_x1), dst_y1 + (src_y2 - src_y1)
    canvas[dst_y1:dst_y2, dst_x1:dst_x2] = frame_bgr[src_y1:src_y2, src_x1:src_x2]
    return cv2.resize(canvas, (image_size, image_size))


def extract_frames_from_video(video_path, num_frames):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    wanted_idxs = np.linspace(0, total - 1, num_frames).astype(int).tolist()
    idx_set = set(wanted_idxs)
    frame_dict = {}
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i in idx_set:
            frame_dict[i] = frame
        i += 1
    cap.release()
    frames = []
    for idx in wanted_idxs:
        if idx in frame_dict:
            frames.append(frame_dict[idx])
        elif frames:
            frames.append(frames[-1])
    if not frames:
        return []
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return frames


def crop_video_to_frames(video_path, out_dir, num_frames=NUM_FRAMES_CACHED,
                          margin_jitter=0):
    """Extract + MTCNN-crop a raw video's frames into out_dir.
    margin_jitter > 0 is a TRAIN-TIME-ONLY augmentation (randomizes crop
    tightness per video so the model can't use exact crop margin as a
    shortcut). Always pass margin_jitter=0 for inference / eval videos."""
    frames = extract_frames_from_video(video_path, num_frames)
    if not frames:
        return 0

    margin = MTCNN_MARGIN
    if margin_jitter > 0:
        margin = max(0, MTCNN_MARGIN + _random.randint(-margin_jitter, margin_jitter))

    os.makedirs(out_dir, exist_ok=True)
    n_written = 0
    for i, frame in enumerate(frames):
        crop = crop_face_mtcnn(frame, mode=MTCNN_CROP_MODE, image_size=IMG_SIZE, margin=margin)
        cv2.imwrite(os.path.join(out_dir, f"frame_{i:03d}.jpg"), crop)
        n_written += 1
    return n_written

print("MTCNN cropping functions ready.")


## 4. Index samples

Each video folder under `FACE_ROOT` becomes one sample carrying its label, sub-category, data
source, and an `identity_key` used for the leakage-safe split.


In [ ]:
def _list_video_dirs(path):
    if not os.path.isdir(path):
        return []
    return sorted([d for d in Path(path).iterdir() if d.is_dir()])


def _has_frame_files(path):
    return any(
        f.is_file() and f.suffix.lower() in FRAME_EXTS
        for f in Path(path).iterdir()
    )


def _frame_count(path):
    return sum(
        1 for f in Path(path).iterdir()
        if f.is_file() and f.suffix.lower() in FRAME_EXTS
    )


def build_samples_from_face_root(face_root, source_tag):
    samples = []

    def add_sample(video_dir, label_name, sub_category, identity):
        if _frame_count(video_dir) == 0:
            return
        samples.append({
            "video_id": video_dir.name,
            "folder_path": str(video_dir),
            "label": LABEL_MAP[label_name],
            "sub_category": sub_category,
            "source": source_tag,
            "identity_key": f"{source_tag}:{identity}",
        })

    real_root = Path(face_root) / REAL_DIRNAME
    for video_dir in _list_video_dirs(real_root):
        # Real folders are normally plain source IDs (for example, "000").
        add_sample(
            video_dir,
            "Real",
            "Real",
            video_dir.name.split("_")[0],
        )

    fake_root = Path(face_root) / FAKE_DIRNAME
    if fake_root.is_dir():
        for child_dir in _list_video_dirs(fake_root):
            if _has_frame_files(child_dir):
                # Flat layout: Fake/<video_id>/*.jpg
                video_id = child_dir.name
                add_sample(
                    child_dir,
                    "Fake",
                    "Fake",
                    video_id.split("_")[0],
                )
                continue

            # Nested layout: Fake/<method>/<video_id>/*.jpg
            for video_dir in _list_video_dirs(child_dir):
                video_id = video_dir.name
                add_sample(
                    video_dir,
                    "Fake",
                    child_dir.name,
                    video_id.split("_")[0],
                )

    return samples


face_root_samples = build_samples_from_face_root(FACE_ROOT, source_tag="face_root")
print(f"FACE_ROOT samples: {len(face_root_samples)}")
print(Counter(s["label"] for s in face_root_samples))

# Single dataset — no separate raw-video source to merge
all_samples = face_root_samples
print(f"Total samples: {len(all_samples)}")
print("By label:", Counter(s["label"] for s in all_samples))

sample_fake_names = [s["video_id"] for s in face_root_samples if s["label"] == 1][:10]
print(sample_fake_names) 

In [ ]:
# Single dataset — no separate raw-video source to merge
all_samples = face_root_samples
print(f"Total samples: {len(all_samples)}")
print("By label:", Counter([s["label"] for s in all_samples]))

sample_fake_names = [s["video_id"] for s in face_root_samples if s["label"]==1][:10]
print(sample_fake_names)  # confirm format is like "023_041" (source_target)

## 5. Identity-aware, source-stratified split

Random per-video splitting leaks identity (the same person appears in a Real video and every Fake
derived from it). We split by `identity_key` with `GroupShuffleSplit` so all videos sharing an
identity land in the same split, and keep sources balanced across splits. Class weights for the
loss are computed on the TRAIN split only.


In [ ]:
def group_split(samples, test_size, seed):
    groups = [s["identity_key"] for s in samples]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(samples))
    train_idx, test_idx = next(gss.split(idx, groups=groups))
    return [samples[i] for i in train_idx], [samples[i] for i in test_idx]

# Split per-source first (so both sources are represented in val/test),
# then merge -- this avoids one dataset accidentally being 100% train.
train_samples, val_samples, test_samples = [], [], []
for source_tag in set(s["source"] for s in all_samples):
    src_samples = [s for s in all_samples if s["source"] == source_tag]
    tr, temp = group_split(src_samples, VAL_SPLIT + TEST_SPLIT, SEED)
    va, te = group_split(temp, TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT), SEED)
    train_samples += tr; val_samples += va; test_samples += te

random.shuffle(train_samples); random.shuffle(val_samples); random.shuffle(test_samples)

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")
print("Train labels:", Counter([s["label"] for s in train_samples]))
print("Val labels:  ", Counter([s["label"] for s in val_samples]))
print("Test labels: ", Counter([s["label"] for s in test_samples]))
print("Test sources:", Counter([s["source"] for s in test_samples]))

SPLITS = {"train": train_samples, "val": val_samples, "test": test_samples}

# Sanity check: no identity should appear in more than one split
train_ids = set(s["identity_key"] for s in train_samples)
val_ids = set(s["identity_key"] for s in val_samples)
test_ids = set(s["identity_key"] for s in test_samples)
assert not (train_ids & val_ids), "Identity leakage between train/val!"
assert not (train_ids & test_ids), "Identity leakage between train/test!"
assert not (val_ids & test_ids), "Identity leakage between val/test!"
print("No identity leakage across splits confirmed.")

# Class weights for the loss functions (computed on TRAIN split only)
train_labels = [s["label"] for s in train_samples]
n_real, n_fake = train_labels.count(0), train_labels.count(1)
CLASS_WEIGHTS = torch.tensor(
    [len(train_labels) / (2 * n_real), len(train_labels) / (2 * n_fake)],
    dtype=torch.float32,
).to(device)
print("Class weights [Real, Fake]:", CLASS_WEIGHTS.tolist())


## 6. Exploratory Data Analysis (EDA)

A complete pass over the dataset **before** training:
1. Class distribution — counts and percentages.
2. Per-video frame counts, frame resolution, and FPS / duration.
3. Sample frame-sequence grids (Real vs Fake).
4. Data-health checks — empty/corrupt folders, unreadable frames, missing frames.

Scans are capped by `EDA_MAX_VIDEOS` / `EDA_HEALTH_SCAN_LIMIT` so EDA stays fast on ~22k videos.


In [ ]:
# ------------------------------------------------------------
# EDA 1 — Class distribution (counts & percentages)
# ------------------------------------------------------------
label_counts = Counter(s["label"] for s in all_samples)
total = len(all_samples)

dist_rows = []
for label_id in sorted(label_counts):
    n = label_counts[label_id]
    dist_rows.append({
        "class": ID2LABEL[label_id], "label_id": label_id,
        "count": n, "percent": 100.0 * n / total,
    })
class_dist = pd.DataFrame(dist_rows)
print(f"Total videos: {total}")
print(class_dist.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Per-source breakdown (informative when more than one dataset is merged)
src_rows = []
for src in sorted(set(s["source"] for s in all_samples)):
    sub = [s for s in all_samples if s["source"] == src]
    c = Counter(s["label"] for s in sub)
    src_rows.append({"source": src, "Real": c.get(0, 0), "Fake": c.get(1, 0), "total": len(sub)})
print("\nBy source:")
print(pd.DataFrame(src_rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(class_dist["class"], class_dist["count"], color=["#2ca02c", "#d62728"])
axes[0].set_title("Class distribution (count)"); axes[0].set_ylabel("Videos")
for i, (n, p) in enumerate(zip(class_dist["count"], class_dist["percent"])):
    axes[0].text(i, n, f"{n}\n({p:.1f}%)", ha="center", va="bottom", fontsize=9)
axes[1].pie(class_dist["count"], labels=class_dist["class"], autopct="%1.1f%%",
            colors=["#2ca02c", "#d62728"], startangle=90)
axes[1].set_title("Class balance (%)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eda_class_distribution.png"), dpi=150)
plt.show()


In [ ]:
# ------------------------------------------------------------
# EDA 2 — Frame counts, resolution, FPS & duration
# ------------------------------------------------------------
def _list_frame_files(folder):
    return sorted(f for f in Path(folder).glob("*") if f.suffix.lower() in FRAME_EXTS)

eda_sample = all_samples if len(all_samples) <= EDA_MAX_VIDEOS else random.sample(all_samples, EDA_MAX_VIDEOS)
print(f"Scanning {len(eda_sample)} / {len(all_samples)} videos for frame-count & resolution stats...")

frame_counts, widths, heights, bad_read = [], [], [], 0
for s in tqdm(eda_sample, desc="EDA: frames/resolution"):
    files = _list_frame_files(s["folder_path"])
    frame_counts.append(len(files))
    if files:
        img = cv2.imread(str(files[0]))          # read only the first frame (fast)
        if img is None:
            bad_read += 1
        else:
            h, w = img.shape[:2]
            heights.append(h); widths.append(w)

frame_counts = np.array(frame_counts)
print("\n--- Frames per video ---")
print(f"min {frame_counts.min()} | max {frame_counts.max()} | mean {frame_counts.mean():.2f} | "
      f"median {np.median(frame_counts):.1f}")
print("distinct frame counts:", dict(sorted(Counter(frame_counts.tolist()).items())))

if widths:
    print("\n--- Frame resolution (W x H) ---")
    for (w, h), n in Counter(zip(widths, heights)).most_common(10):
        print(f"  {w}x{h}: {n} videos")
print(f"\nFirst-frame read failures during scan: {bad_read}")

# --- FPS / duration ---
# Still-frame folders do NOT carry FPS or wall-clock duration. If RAW_VIDEO_ROOT is set we read
# true FPS/duration from the source videos; otherwise we report a nominal estimate so the number
# is never silently fabricated.
fps_vals, dur_vals = [], []
if RAW_VIDEO_ROOT and os.path.isdir(RAW_VIDEO_ROOT):
    vid_exts = (".mp4", ".avi", ".mov", ".mkv", ".webm")
    vids = [p for p in Path(RAW_VIDEO_ROOT).rglob("*") if p.suffix.lower() in vid_exts][:EDA_MAX_VIDEOS]
    for vp in tqdm(vids, desc="EDA: fps/duration"):
        cap = cv2.VideoCapture(str(vp))
        fps = cap.get(cv2.CAP_PROP_FPS); nfrm = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        if fps and fps > 0 and nfrm and nfrm > 0:
            fps_vals.append(fps); dur_vals.append(nfrm / fps)
    if fps_vals:
        print("\n--- FPS / duration (from RAW_VIDEO_ROOT) ---")
        print(f"FPS      mean {np.mean(fps_vals):.2f} | min {np.min(fps_vals):.2f} | max {np.max(fps_vals):.2f}")
        print(f"Duration mean {np.mean(dur_vals):.2f}s | min {np.min(dur_vals):.2f}s | max {np.max(dur_vals):.2f}s")
else:
    print("\n--- FPS / duration ---")
    print("RAW_VIDEO_ROOT not set: FPS/duration are NOT recoverable from pre-cropped still frames.")
    print(f"Nominal estimate @ {EDA_NOMINAL_FPS:.0f} fps -> mean clip duration "
          f"~ {frame_counts.mean() / EDA_NOMINAL_FPS:.2f}s ({frame_counts.mean():.1f} frames/video).")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
nbins = int(min(30, max(5, frame_counts.max() - frame_counts.min() + 1)) or 10)
axes[0].hist(frame_counts, bins=nbins, color="#1f77b4")
axes[0].set_title("Frames per video"); axes[0].set_xlabel("Frame count"); axes[0].set_ylabel("Videos")
if widths:
    axes[1].hist(widths, bins=30, alpha=0.6, label="width", color="#2ca02c")
    axes[1].hist(heights, bins=30, alpha=0.6, label="height", color="#ff7f0e")
    axes[1].set_title("Frame resolution (px)"); axes[1].set_xlabel("Pixels"); axes[1].legend()
else:
    axes[1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "eda_frames_resolution.png"), dpi=150)
plt.show()


In [ ]:
# ------------------------------------------------------------
# EDA 3 — Sample frame-sequence grids (Real vs Fake)
# ------------------------------------------------------------
def _load_grid_rows(samples_by_class, n_videos, n_frames):
    rows = []
    for cls, samples in samples_by_class.items():
        for s in samples[:n_videos]:
            files = _list_frame_files(s["folder_path"])
            if not files:
                continue
            idxs = np.linspace(0, len(files) - 1, min(n_frames, len(files))).astype(int)
            imgs = []
            for i in idxs:
                img = cv2.imread(str(files[i]))
                imgs.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None
                            else np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8))
            rows.append((f"{cls}: {s['video_id']}", imgs))
    return rows

def _show_clip_grid(rows):
    if not rows:
        print("No frames to display."); return
    ncol = max(len(r[1]) for r in rows)
    fig, axes = plt.subplots(len(rows), ncol, figsize=(1.9 * ncol, 2.3 * len(rows)), squeeze=False)
    for r, (title, imgs) in enumerate(rows):
        for c in range(ncol):
            ax = axes[r][c]
            if c < len(imgs):
                ax.imshow(imgs[c])
            ax.set_xticks([]); ax.set_yticks([])
            if c == 0:
                ax.set_title(title, fontsize=8, loc="left")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "eda_sample_grids.png"), dpi=120)
    plt.show()

real_samples = [s for s in all_samples if s["label"] == 0]
fake_samples = [s for s in all_samples if s["label"] == 1]
random.shuffle(real_samples); random.shuffle(fake_samples)
grid_rows = _load_grid_rows(
    {"Real": real_samples, "Fake": fake_samples},
    n_videos=EDA_GRID_VIDEOS_PER_CLASS,
    n_frames=EDA_GRID_FRAMES_PER_VIDEO,
)
print(f"Showing {len(grid_rows)} clips "
      f"({EDA_GRID_FRAMES_PER_VIDEO} sampled frames each, left->right = start->end).")
_show_clip_grid(grid_rows)


In [ ]:
# ------------------------------------------------------------
# EDA 4 — Data-health checks (corrupt / unreadable / missing frames)
# ------------------------------------------------------------
health_sample = all_samples if len(all_samples) <= EDA_HEALTH_SCAN_LIMIT else random.sample(all_samples, EDA_HEALTH_SCAN_LIMIT)
print(f"Health-scanning {len(health_sample)} / {len(all_samples)} videos "
      f"(expected ~{NUM_FRAMES_CACHED} frames/video)...")

empty_folders, zero_byte, unreadable, short_folders = [], [], [], []
for s in tqdm(health_sample, desc="EDA: health"):
    files = _list_frame_files(s["folder_path"])
    if len(files) == 0:
        empty_folders.append(s["video_id"]); continue
    if len(files) < NUM_FRAMES_CACHED:
        short_folders.append((s["video_id"], len(files)))
    for f in files:
        try:
            if f.stat().st_size == 0:
                zero_byte.append(str(f))
        except OSError:
            zero_byte.append(str(f))
    # cheap integrity probe: decode the first and last frame of each clip
    for f in (files[0], files[-1]):
        if cv2.imread(str(f)) is None:
            unreadable.append(str(f))

health_summary = pd.DataFrame([
    {"check": "videos scanned",                    "value": len(health_sample)},
    {"check": "empty folders (0 frames)",          "value": len(empty_folders)},
    {"check": "folders with < expected frames",    "value": len(short_folders)},
    {"check": "zero-byte / stat-failed files",     "value": len(zero_byte)},
    {"check": "unreadable probe frames (None)",    "value": len(unreadable)},
])
print("\n--- Data-health summary ---")
print(health_summary.to_string(index=False))

if empty_folders:  print("\nExample empty folders:", empty_folders[:10])
if short_folders:  print("Example short folders (id, n_frames):", short_folders[:10])
if unreadable:     print("Example unreadable frames:", unreadable[:5])
if zero_byte:      print("Example zero-byte files:", zero_byte[:5])

healthy = not (empty_folders or unreadable or zero_byte)
print("\nOverall:",
      "HEALTHY — no corrupt / unreadable / empty samples detected."
      if healthy else
      "ISSUES FOUND — review the lists above before training.")


## 7. Swin3D dataset — uniform 24-frame temporal sampling

Each clip is sampled to `num_frames` frames (default `NUM_FRAMES_SWIN = 24`) and returned as
`(C, T, H, W)` — the layout torchvision video models expect — then batched to `(B, C, 24, H, W)`.
The frame count is a constructor argument so the *same* class also builds the 32-frame loader used
by the comparison stage. Kinetics-400 preprocessing comes from the weights' own `.transforms()`, so
it can never drift from what the pretrained backbone expects.


In [ ]:
from torchvision.models.video import swin3d_t, swin3d_s, swin3d_b, Swin3D_T_Weights, Swin3D_S_Weights, Swin3D_B_Weights
from PIL import Image as PILImage

# Maps SWIN_VARIANT -> (constructor, Kinetics-400 pretrained weights enum). Reused by the model
# cell and mirrored in predict_video.py so train/inference preprocessing never diverge.
SWIN_BUILDERS = {
    "swin3d_t": (swin3d_t, Swin3D_T_Weights.KINETICS400_V1),
    "swin3d_s": (swin3d_s, Swin3D_S_Weights.KINETICS400_V1),
    "swin3d_b": (swin3d_b, Swin3D_B_Weights.KINETICS400_V1),
}
_swin_ctor, SWIN_PRETRAINED_WEIGHTS = SWIN_BUILDERS[SWIN_VARIANT]

# Swin3D's own Kinetics-400 preprocessing: resize shortest side 256, center-crop 224x224, rescale
# to [0,1], ImageNet normalize, permute to (C, T, H, W). Using the weights' own transforms
# guarantees parity with the pretrained backbone.
swin_eval_transform = SWIN_PRETRAINED_WEIGHTS.transforms()

def uniform_sample_indices_with_pad(total, k):
    """Return a list of (frame_index, is_padded) pairs of length k.

    If >= k real frames exist, uniformly subsample them (no padding). If fewer exist, every real
    frame is used once and the shortfall is padded by repeating the LAST real frame's index,
    flagged is_padded=True so the caller can noise-perturb it instead of inserting an exact dup.
    With NUM_FRAMES_CACHED=30 and k=24 no padding occurs; with k=32 (the pretrained comparison
    model) 2 frames are padded.
    """
    if total >= k:
        idxs = np.linspace(0, total - 1, k).astype(int).tolist()
        return [(i, False) for i in idxs]
    real = [(i, False) for i in range(total)]
    padded = [(total - 1, True)] * (k - total)
    return real + padded

def _augment_pad_frame(frame_chw_uint8, noise_std, rng):
    """Perturb a duplicated pad frame with Gaussian pixel noise so it isn't an exact copy of the
    last real frame. rng is a np.random.RandomState for reproducible eval, or None to use the
    global (stochastic) np.random state during training."""
    noise_fn = np.random.normal if rng is None else rng.normal
    noise = noise_fn(0, noise_std, size=tuple(frame_chw_uint8.shape)).astype(np.float32)
    out = frame_chw_uint8.to(torch.float32) + torch.from_numpy(noise)
    return out.clamp(0, 255).to(torch.uint8)

class VideoClipDataset(Dataset):
    """Returns pixel_values shaped (C, T, H, W) for torchvision video models (incl. Swin3D).

    num_frames is configurable so one class serves both the 24-frame training/eval pipeline and
    the 32-frame pretrained-comparison pipeline."""
    def __init__(self, samples, train=False, num_frames=NUM_FRAMES_SWIN):
        self.samples = samples
        self.train = train
        self.num_frames = num_frames
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        frame_files = sorted([f for f in Path(s["folder_path"]).glob("*") if f.suffix.lower() in FRAME_EXTS])[:NUM_FRAMES_CACHED]
        idx_flags = uniform_sample_indices_with_pad(len(frame_files), self.num_frames)

        frames = []
        pad_ctr = 0
        for i, is_pad in idx_flags:
            img = PILImage.open(frame_files[i]).convert("RGB")
            if self.train and random.random() < 0.5:
                img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
            frame_t = torch.from_numpy(np.array(img)).permute(2, 0, 1)  # (C, H, W) uint8
            if is_pad:
                # Train: fresh random noise each epoch. Val/test: fixed seed per pad slot (reproducible).
                rng = None if self.train else np.random.RandomState(1000 + pad_ctr)
                frame_t = _augment_pad_frame(frame_t, SWIN_PAD_NOISE_STD, rng)
                pad_ctr += 1
            frames.append(frame_t)

        clip = torch.stack(frames, dim=0)          # (T, C, H, W) uint8
        pixel_values = swin_eval_transform(clip)   # -> (C, T, H, W) float, normalized
        return {"pixel_values": pixel_values, "labels": torch.tensor(s["label"], dtype=torch.long)}

def swin_collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }

swin_datasets = {
    "train": VideoClipDataset(SPLITS["train"], train=True,  num_frames=NUM_FRAMES_SWIN),
    "val":   VideoClipDataset(SPLITS["val"],   train=False, num_frames=NUM_FRAMES_SWIN),
    "test":  VideoClipDataset(SPLITS["test"],  train=False, num_frames=NUM_FRAMES_SWIN),
}
print("Swin3D datasets (num_frames = %d):" % NUM_FRAMES_SWIN, {k: len(v) for k, v in swin_datasets.items()})


## 8. Swin3D model — Kinetics-400 backbone + LoRA + Linear head

Backbone: `swin3d_t` with its Kinetics-400 classifier stripped to a pooled embedding. We attach a
`Linear(embed_dim, 2)` head, wrap with PEFT LoRA on `qkv`/`proj`, and (since LoRA alone underfits)
unfreeze the final Swin stage + norm for low-LR fine-tuning. This is the **entire** model — there is
no second stream and no fusion head.


In [ ]:
!pip uninstall torchao -y

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache(); gc.collect()

# --- Backbone: Video Swin Transformer (3D), Kinetics-400 pretrained ---
swin_backbone = _swin_ctor(weights=SWIN_PRETRAINED_WEIGHTS)
SWIN_EMBED_DIM = swin_backbone.head.in_features
swin_backbone.head = nn.Identity()   # strip the Kinetics-400 classifier -> pooled embedding


In [ ]:

class SwinClassifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, pixel_values, labels=None):
        emb = self.backbone(pixel_values)
        return self.classifier(emb)

base_swin = SwinClassifier(swin_backbone, SWIN_EMBED_DIM, len(LABEL_MAP))

from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=SWIN_LORA_RANK, lora_alpha=SWIN_LORA_ALPHA, lora_dropout=SWIN_LORA_DROPOUT, bias="none",
    target_modules=SWIN_TARGET_MODULES,
    modules_to_save=["classifier"],
)
swin_model = get_peft_model(base_swin, lora_config).to(device)

if SWIN_UNFREEZE_LAST_STAGE:
    # LoRA-only training was underfitting. Fine-tune the final Swin stage and norm
    # with a smaller LR while keeping the rest frozen.
    last_stage = swin_model.base_model.model.backbone.features[-1]
    for p in last_stage.parameters():
        p.requires_grad = True
    for p in swin_model.base_model.model.backbone.norm.parameters():
        p.requires_grad = True

swin_model.print_trainable_parameters()

## 9. Train Swin3D (24 frames) — LoRA + last-stage fine-tune, class-weighted loss

Real training loop: AdamW with two LR groups (higher for the LoRA adapter + head, lower for the
unfrozen backbone stage), cosine-annealed LR, class-weighted cross-entropy with label smoothing,
AMP, gradient accumulation, and gradient clipping. We checkpoint on `0.5*val_acc + 0.5*val_macroF1`
with early stopping, then plot the real learning curves. The best checkpoint is written to
`OUTPUT_DIR/swin_lora_best` and reloaded for evaluation.


In [ ]:
swin_train_loader = DataLoader(swin_datasets["train"], batch_size=SWIN_BATCH_SIZE, shuffle=True, collate_fn=swin_collate_fn, num_workers=0)
swin_val_loader = DataLoader(swin_datasets["val"], batch_size=SWIN_BATCH_SIZE, shuffle=False, collate_fn=swin_collate_fn, num_workers=0)
swin_test_loader = DataLoader(swin_datasets["test"], batch_size=SWIN_BATCH_SIZE, shuffle=False, collate_fn=swin_collate_fn, num_workers=0)

backbone_param_ids = set()
if SWIN_UNFREEZE_LAST_STAGE:
    for p in swin_model.base_model.model.backbone.features[-1].parameters():
        backbone_param_ids.add(id(p))
    for p in swin_model.base_model.model.backbone.norm.parameters():
        backbone_param_ids.add(id(p))

backbone_params, adapter_head_params = [], []
for p in swin_model.parameters():
    if not p.requires_grad:
        continue
    (backbone_params if id(p) in backbone_param_ids else adapter_head_params).append(p)

param_groups = [{"params": adapter_head_params, "lr": SWIN_LR}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": SWIN_BACKBONE_LR})

optimizer = torch.optim.AdamW(param_groups, weight_decay=SWIN_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SWIN_EPOCHS)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=SWIN_LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_epoch(model, loader, optimizer, criterion, device, train, grad_accum=1):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    if train:
        optimizer.zero_grad(set_to_none=True)
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for step, batch in enumerate(tqdm(loader, desc="train" if train else "eval", leave=False)):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(pixel_values=pixel_values)
                loss = criterion(logits, labels)
            if train:
                scaler.scale(loss / grad_accum).backward()
                do_step = ((step + 1) % grad_accum == 0) or ((step + 1) == len(loader))
                if do_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).detach().cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / max(1, len(loader)), acc, f1, macro_f1


In [ ]:
# ------------------------------------------------------------
# Real Swin3D training loop (24-frame model)
# ------------------------------------------------------------
best_val_score = -float("inf")
best_val_loss = float("inf")
patience_ctr = 0
swin_ckpt = os.path.join(OUTPUT_DIR, "swin_lora_best")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

def save_swin_trainable_backbone(model, out_dir):
    """Persist the manually-unfrozen backbone weights (final stage + norm). PEFT's save_pretrained
    only stores the adapter deltas + modules_to_save, NOT these, so they are saved separately."""
    if not SWIN_UNFREEZE_LAST_STAGE:
        return
    os.makedirs(out_dir, exist_ok=True)
    torch.save(
        {
            "last_stage": model.base_model.model.backbone.features[-1].state_dict(),
            "norm": model.base_model.model.backbone.norm.state_dict(),
        },
        os.path.join(out_dir, "swin_finetuned_backbone.pt"),
    )

set_deterministic(SEED)  # re-seed so the epoch/shuffle order is reproducible
for epoch in range(1, SWIN_EPOCHS + 1):
    train_loss, train_acc, train_f1, train_macro_f1 = run_epoch(
        swin_model, swin_train_loader, optimizer, criterion, device, train=True, grad_accum=SWIN_GRAD_ACCUM
    )
    val_loss, val_acc, val_f1, val_macro_f1 = run_epoch(
        swin_model, swin_val_loader, optimizer, criterion, device, train=False
    )
    scheduler.step()

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    val_score = 0.5 * val_acc + 0.5 * val_macro_f1
    print(f"[Swin3D] Epoch {epoch}/{SWIN_EPOCHS} | train_loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val_loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} macroF1 {val_macro_f1:.4f}")

    if (val_score > best_val_score) or (abs(val_score - best_val_score) < 1e-6 and val_loss < best_val_loss):
        best_val_score, best_val_loss, patience_ctr = val_score, val_loss, 0
        swin_model.save_pretrained(swin_ckpt)
        save_swin_trainable_backbone(swin_model, swin_ckpt)
        print(f"  Saved new best Swin3D checkpoint (score={best_val_score:.4f}, val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= SWIN_PATIENCE:
            print("Early stopping Swin3D."); break

print(f"Swin3D training complete. Best val score: {best_val_score:.4f} | best val_loss: {best_val_loss:.4f}")

# --- Real learning curves ---
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_range, history["train_loss"], label="Train Loss", color="#1f77b4", linewidth=2)
ax1.plot(epochs_range, history["val_loss"], label="Val Loss", color="#ff7f0e", linewidth=2, linestyle="--")
ax1.set_title("Swin3D Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(epochs_range, history["train_acc"], label="Train Acc", color="#2ca02c", linewidth=2)
ax2.plot(epochs_range, history["val_acc"], label="Val Acc", color="#d62728", linewidth=2, linestyle="--")
ax2.set_title(f"Swin3D Accuracy (best val score {best_val_score:.4f})")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "swin_learning_curves.png"), dpi=150)
plt.show()


## 10. Evaluation helpers (Swin3D-only)

Model-agnostic utilities used by the comparison and final reporting: full classification metrics
(accuracy, precision, recall, F1, AUC-ROC), a checkpoint loader that rebuilds a Swin3D + LoRA model
from disk, a probability-returning evaluator, and an inference-cost probe (ms/video and peak GPU VRAM).


In [ ]:
from peft import PeftModel

def full_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); y_prob = np.asarray(y_prob)
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob) if len(set(y_true.tolist())) > 1 else float("nan"),
    }

def build_swin_model_from_ckpt(ckpt_dir):
    """Rebuild a Swin3D + LoRA classifier from a saved checkpoint directory: load the PEFT adapter
    (+ classifier via modules_to_save) and, if present, the fine-tuned backbone last-stage/norm."""
    backbone = _swin_ctor(weights=SWIN_PRETRAINED_WEIGHTS)
    backbone.head = nn.Identity()
    base = SwinClassifier(backbone, SWIN_EMBED_DIM, len(LABEL_MAP))
    model = PeftModel.from_pretrained(base, ckpt_dir)
    bb_path = os.path.join(ckpt_dir, "swin_finetuned_backbone.pt")
    if os.path.exists(bb_path):
        state = torch.load(bb_path, map_location="cpu")
        model.base_model.model.backbone.features[-1].load_state_dict(state["last_stage"], strict=False)
        model.base_model.model.backbone.norm.load_state_dict(state["norm"], strict=False)
    return model.to(device).eval()

@torch.no_grad()
def evaluate_swin(model, loader):
    """Return (y_true, y_pred, prob_fake) over a loader."""
    model.eval()
    ys, preds, probs = [], [], []
    for batch in loader:
        x = batch["pixel_values"].to(device)
        logits = model(pixel_values=x)
        probs.extend(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
        preds.extend(logits.argmax(1).cpu().numpy())
        ys.extend(batch["labels"].numpy())
    return np.array(ys), np.array(preds), np.array(probs)

@torch.no_grad()
def measure_inference(model, loader, max_videos=None):
    """Return (avg_ms_per_video, peak_vram_gb). CUDA-synchronized for honest timing."""
    model.eval()
    is_cuda = device.type == "cuda"
    if is_cuda:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(device)
    per_video_ms, seen = [], 0
    for batch in loader:
        x = batch["pixel_values"].to(device)
        b = x.shape[0]
        if is_cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(pixel_values=x)
        if is_cuda: torch.cuda.synchronize()
        t1 = time.perf_counter()
        per_video_ms.append((t1 - t0) * 1000.0 / b)
        seen += b
        if max_videos is not None and seen >= max_videos:
            break
    avg_ms = float(np.mean(per_video_ms)) if per_video_ms else float("nan")
    peak_vram = float(torch.cuda.max_memory_allocated(device) / 1e9) if is_cuda else float("nan")
    return avg_ms, peak_vram

print("Evaluation helpers ready.")


## 11. Performance comparison — 32-frame pretrained vs 24-frame (this notebook)

Both models are evaluated on the **identical** `SPLITS["test"]` videos; only the temporal sampling
differs (32 frames with padding for the pretrained model, 24 frames for the model trained above).
We report a metrics table (accuracy, precision, recall, F1, AUC-ROC), side-by-side confusion
matrices, overlaid ROC curves, and compute cost (inference ms/video and peak GPU VRAM).


In [ ]:
# ------------------------------------------------------------
# 11a — Load both models, evaluate on the same test split
# ------------------------------------------------------------
assert RUN_COMPARISON, "Set RUN_COMPARISON=True to run the comparison."
assert os.path.isdir(swin_ckpt), f"Trained 24-frame checkpoint not found at {swin_ckpt} (run section 9)."
assert os.path.isdir(PRETRAINED_32F_CKPT), f"32-frame pretrained checkpoint not found at {PRETRAINED_32F_CKPT}."

# Same test videos, two temporal-sampling configs
test_ds_24 = VideoClipDataset(SPLITS["test"], train=False, num_frames=NUM_FRAMES_SWIN)                 # 24
test_ds_32 = VideoClipDataset(SPLITS["test"], train=False, num_frames=COMPARE_NUM_FRAMES_PRETRAINED)   # 32
loader_24 = DataLoader(test_ds_24, batch_size=SWIN_BATCH_SIZE, shuffle=False, collate_fn=swin_collate_fn, num_workers=0)
loader_32 = DataLoader(test_ds_32, batch_size=SWIN_BATCH_SIZE, shuffle=False, collate_fn=swin_collate_fn, num_workers=0)

# 24-frame model trained above (best checkpoint) + existing 32-frame pretrained model
model_24 = build_swin_model_from_ckpt(swin_ckpt)
model_32 = build_swin_model_from_ckpt(PRETRAINED_32F_CKPT)

y24, pred24, prob24 = evaluate_swin(model_24, loader_24)
y32, pred32, prob32 = evaluate_swin(model_32, loader_32)
assert np.array_equal(y24, y32), "Test label order differs between loaders!"

ms24, vram24 = measure_inference(model_24, loader_24, max_videos=COMPARE_MAX_TIMING_VIDEOS)
ms32, vram32 = measure_inference(model_32, loader_32, max_videos=COMPARE_MAX_TIMING_VIDEOS)

rows = []
for name, y, p, pr, ms, vram, nf in [
    ("32-frame pretrained",      y32, pred32, prob32, ms32, vram32, COMPARE_NUM_FRAMES_PRETRAINED),
    ("24-frame (this notebook)", y24, pred24, prob24, ms24, vram24, NUM_FRAMES_SWIN),
]:
    m = full_metrics(y, p, pr)
    rows.append({
        "Model": name, "Frames": nf,
        "Accuracy": m["accuracy"], "Precision": m["precision"], "Recall": m["recall"],
        "F1": m["f1"], "AUC-ROC": m["roc_auc"],
        "Inference (ms/video)": ms, "Peak VRAM (GB)": vram,
    })
comparison_table = pd.DataFrame(rows)

print("=" * 96)
print(f"PERFORMANCE COMPARISON — identical test split (n={len(y24)})")
print("=" * 96)
print(comparison_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- 32-frame pretrained: classification report ---")
print(classification_report(y32, pred32, target_names=["Real", "Fake"], digits=4))
print("--- 24-frame model: classification report ---")
print(classification_report(y24, pred24, target_names=["Real", "Fake"], digits=4))

comparison_table.to_csv(os.path.join(OUTPUT_DIR, "comparison_metrics.csv"), index=False)
print("Saved comparison table ->", os.path.join(OUTPUT_DIR, "comparison_metrics.csv"))


In [ ]:
# ------------------------------------------------------------
# 11b — Side-by-side confusion matrices + overlaid ROC curves + compute cost
# ------------------------------------------------------------
cm32 = confusion_matrix(y32, pred32)
cm24 = confusion_matrix(y24, pred24)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, cm, title in [(axes[0], cm32, "32-frame pretrained"),
                      (axes[1], cm24, "24-frame (this notebook)")]:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
plt.suptitle("Confusion matrices — identical test split", y=1.03, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 6))
for y, pr, label, color in [(y32, prob32, "32-frame pretrained", "#d62728"),
                            (y24, prob24, "24-frame (this notebook)", "#1f77b4")]:
    fpr, tpr, _ = roc_curve(y, pr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{label} (AUC={roc_auc_score(y, pr):.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — 32-frame pretrained vs 24-frame"); ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_roc.png"), dpi=150)
plt.show()

# --- Compute-cost summary ---
speed_df = pd.DataFrame([
    {"Model": "32-frame pretrained",      "Frames": COMPARE_NUM_FRAMES_PRETRAINED, "Inference (ms/video)": ms32, "Peak VRAM (GB)": vram32},
    {"Model": "24-frame (this notebook)", "Frames": NUM_FRAMES_SWIN,               "Inference (ms/video)": ms24, "Peak VRAM (GB)": vram24},
])
print("--- Compute cost ---")
print(speed_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
if not (math.isnan(ms32) or math.isnan(ms24)) and ms24 > 0:
    print(f"\n24-frame vs 32-frame: {ms32 / ms24:.2f}x faster per video ({ms32 - ms24:+.1f} ms/video).")
if not (math.isnan(vram32) or math.isnan(vram24)):
    print(f"Peak VRAM delta (24f - 32f): {vram24 - vram32:+.3f} GB.")


## 12. Save artifacts (Swin3D-only)

Promote the best 24-frame checkpoint to `swin_lora_final`, and write the crop + model configs that
`predict_video.py` needs so inference preprocessing and frame count match training exactly.


In [ ]:
# Promote the best checkpoint -> final (deployable) directory
swin_final_dir = os.path.join(OUTPUT_DIR, "swin_lora_final")
if os.path.isdir(swin_final_dir):
    shutil.rmtree(swin_final_dir)
if os.path.isdir(swin_ckpt):
    shutil.copytree(swin_ckpt, swin_final_dir)
    print(f"Promoted best checkpoint -> {swin_final_dir}")
else:
    print(f"Warning: best checkpoint '{swin_ckpt}' not found; skipping promotion.")

with open(os.path.join(OUTPUT_DIR, "mtcnn_config.json"), "w") as f:
    json.dump({
        "mode": MTCNN_CROP_MODE, "margin": MTCNN_MARGIN,
        "min_face_size": MTCNN_MIN_FACE_SIZE, "thresholds": MTCNN_THRESHOLDS,
    }, f)

with open(os.path.join(OUTPUT_DIR, "swin_config.json"), "w") as f:
    json.dump({
        "variant": SWIN_VARIANT,
        "embed_dim": SWIN_EMBED_DIM,
        "num_frames": NUM_FRAMES_SWIN,            # 24
        "num_frames_cached": NUM_FRAMES_CACHED,
        "pad_noise_std": SWIN_PAD_NOISE_STD,
        "unfreeze_last_stage": SWIN_UNFREEZE_LAST_STAGE,
    }, f)

print("Saved artifacts to", OUTPUT_DIR)
print(sorted(os.listdir(OUTPUT_DIR)))


## 13. `predict_video.py` — Swin3D-only inference

Standalone script written to disk. Crops faces with the SAME MTCNN pipeline used for training,
samples the SAME frame count (`num_frames` from `swin_config.json`), rebuilds Swin3D + LoRA, and
classifies directly from the `Linear` head (softmax → argmax). No mediapipe, landmarks, ST-GCN,
fusion, or temperature scaling.


In [ ]:
%%writefile predict_video.py
"""
Deepfake Detection - Swin3D-only inference
==========================================
Crops faces with the SAME MTCNN pipeline used to build the training data
(see mtcnn_config.json), samples the SAME frame count (swin_config.json),
rebuilds the Video Swin Transformer (Swin3D, Kinetics-400 pretrained + LoRA)
and classifies directly from its Linear head. No ST-GCN / fusion / mediapipe.

Usage:
    python predict_video.py --video path/to/video.mp4 --checkpoint-dir /kaggle/working/outputs
"""

import os
import json
import argparse
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from facenet_pytorch import MTCNN

IMG_SIZE = 224
LABEL_MAP = {"Real": 0, "Fake": 1}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 1. Frame extraction + MTCNN face cropping (matches training exactly)
# ============================================================
def extract_frames(video_path, num_frames):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read frame count from {video_path}")
    wanted_idxs = np.linspace(0, total - 1, num_frames).astype(int).tolist()
    idx_set = set(wanted_idxs)
    frame_dict, i = {}, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i in idx_set:
            frame_dict[i] = frame
        i += 1
    cap.release()
    frames = []
    for idx in wanted_idxs:
        if idx in frame_dict:
            frames.append(frame_dict[idx])
        elif frames:
            frames.append(frames[-1])
    if not frames:
        raise RuntimeError(f"No frames could be read from {video_path}")
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return frames


_mtcnn_singleton = None
def _get_mtcnn(mtcnn_cfg):
    global _mtcnn_singleton
    if _mtcnn_singleton is None:
        _mtcnn_singleton = MTCNN(
            image_size=IMG_SIZE, margin=0,
            min_face_size=mtcnn_cfg["min_face_size"],
            thresholds=mtcnn_cfg["thresholds"],
            post_process=False, select_largest=True,
            device=device, keep_all=False,
        )
    return _mtcnn_singleton


def _expand_box_pad(box, margin, w, h):
    x1, y1, x2, y2 = box
    x1, y1, x2, y2 = x1 - margin, y1 - margin, x2 + margin, y2 + margin
    bw, bh = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(bw, bh)
    return cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2


def crop_face(frame_bgr, mtcnn_cfg, image_size=IMG_SIZE):
    """Deterministic MTCNN crop - no margin jitter at inference time."""
    h, w = frame_bgr.shape[:2]
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)

    mtcnn = _get_mtcnn(mtcnn_cfg)
    boxes, probs = mtcnn.detect(pil_img)
    if boxes is None or len(boxes) == 0:
        return cv2.resize(frame_bgr, (image_size, image_size))

    box = boxes[0]
    margin = mtcnn_cfg["margin"]
    x1, y1, x2, y2 = _expand_box_pad(box, margin, w, h)
    side = int(round(x2 - x1))
    if side <= 0:
        return cv2.resize(frame_bgr, (image_size, image_size))

    x1_i, y1_i = int(np.floor(x1)), int(np.floor(y1))
    x2_i, y2_i = x1_i + side, y1_i + side
    canvas = np.zeros((side, side, 3), dtype=np.uint8)
    src_x1, src_y1 = max(x1_i, 0), max(y1_i, 0)
    src_x2, src_y2 = min(x2_i, w), min(y2_i, h)
    if src_x2 <= src_x1 or src_y2 <= src_y1:
        return cv2.resize(frame_bgr, (image_size, image_size))

    dst_x1, dst_y1 = src_x1 - x1_i, src_y1 - y1_i
    dst_x2, dst_y2 = dst_x1 + (src_x2 - src_x1), dst_y1 + (src_y2 - src_y1)
    canvas[dst_y1:dst_y2, dst_x1:dst_x2] = frame_bgr[src_y1:src_y2, src_x1:src_x2]
    return cv2.resize(canvas, (image_size, image_size))


def prepare_face_frames(video_path, num_frames, mtcnn_cfg):
    raw_frames = extract_frames(video_path, num_frames)
    return [crop_face(f, mtcnn_cfg) for f in raw_frames]


# ============================================================
# 2. Model definition (must match training exactly)
# ============================================================
class SwinClassifier(nn.Module):
    """Video Swin Transformer backbone (pixel_values -> pooled embedding) + Linear head."""
    def __init__(self, backbone, embed_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(embed_dim, num_classes)
    def forward(self, pixel_values, labels=None):
        emb = self.backbone(pixel_values)
        return self.classifier(emb)


# ============================================================
# 3. Temporal sampling (mirrors training-time padding scheme)
# ============================================================
def _uniform_sample_indices_with_pad(total, k):
    if total >= k:
        idxs = np.linspace(0, total - 1, k).astype(int).tolist()
        return [(i, False) for i in idxs]
    real = [(i, False) for i in range(total)]
    padded = [(total - 1, True)] * (k - total)
    return real + padded


def _augment_pad_frame(frame_chw_uint8, noise_std, rng):
    """Deterministic (fixed-seed) noise for padded frames - inference must be reproducible."""
    noise = rng.normal(0, noise_std, size=tuple(frame_chw_uint8.shape)).astype(np.float32)
    out = frame_chw_uint8.to(torch.float32) + torch.from_numpy(noise)
    return out.clamp(0, 255).to(torch.uint8)


# ============================================================
# 4. Load pipeline
# ============================================================
def load_pipeline(checkpoint_dir):
    from torchvision.models.video import swin3d_t, swin3d_s, swin3d_b, Swin3D_T_Weights, Swin3D_S_Weights, Swin3D_B_Weights
    from peft import PeftModel

    with open(os.path.join(checkpoint_dir, "mtcnn_config.json")) as f:
        mtcnn_cfg = json.load(f)
    with open(os.path.join(checkpoint_dir, "swin_config.json")) as f:
        swin_cfg = json.load(f)

    swin_embed_dim = swin_cfg["embed_dim"]
    num_frames_swin = swin_cfg["num_frames"]
    num_frames_cached = swin_cfg.get("num_frames_cached", max(num_frames_swin, 30))
    pad_noise_std = swin_cfg.get("pad_noise_std", 3.0)

    swin_builders = {
        "swin3d_t": (swin3d_t, Swin3D_T_Weights.KINETICS400_V1),
        "swin3d_s": (swin3d_s, Swin3D_S_Weights.KINETICS400_V1),
        "swin3d_b": (swin3d_b, Swin3D_B_Weights.KINETICS400_V1),
    }
    swin_ctor, swin_weights = swin_builders[swin_cfg["variant"]]
    # PEFT/LoRA checkpoints store only the adapter deltas + classifier head, not the frozen
    # backbone, so rebuild the base backbone from the same Kinetics-400 pretrained weights.
    swin_backbone = swin_ctor(weights=swin_weights)
    swin_backbone.head = nn.Identity()
    swin_eval_transform = swin_weights.transforms()

    base_swin = SwinClassifier(swin_backbone, swin_embed_dim, len(LABEL_MAP))
    swin_dir = os.path.join(checkpoint_dir, "swin_lora_final")
    if not os.path.isdir(swin_dir):
        swin_dir = os.path.join(checkpoint_dir, "swin_lora_best")
    backbone_state_path = os.path.join(swin_dir, "swin_finetuned_backbone.pt")
    if os.path.exists(backbone_state_path):
        state = torch.load(backbone_state_path, map_location="cpu")
        base_swin.backbone.features[-1].load_state_dict(state["last_stage"], strict=False)
        base_swin.backbone.norm.load_state_dict(state["norm"], strict=False)
    swin_model = PeftModel.from_pretrained(base_swin, swin_dir).to(device)
    swin_model.eval()

    return {
        "swin_model": swin_model,
        "swin_eval_transform": swin_eval_transform,
        "num_frames_swin": num_frames_swin,
        "num_frames_cached": num_frames_cached,
        "pad_noise_std": pad_noise_std,
        "mtcnn_cfg": mtcnn_cfg,
    }


# ============================================================
# 5. Predict on a single video
# ============================================================
@torch.no_grad()
def predict_video(video_path, pipeline):
    face_frames = prepare_face_frames(video_path, pipeline["num_frames_cached"], pipeline["mtcnn_cfg"])

    num_frames_swin = pipeline["num_frames_swin"]
    idx_flags = _uniform_sample_indices_with_pad(len(face_frames), num_frames_swin)
    frames_rgb_t, pad_ctr = [], 0
    for i, is_pad in idx_flags:
        f_rgb = cv2.cvtColor(face_frames[i], cv2.COLOR_BGR2RGB)
        frame_t = torch.from_numpy(f_rgb).permute(2, 0, 1)  # (C, H, W) uint8
        if is_pad:
            rng = np.random.RandomState(1000 + pad_ctr)
            frame_t = _augment_pad_frame(frame_t, pipeline["pad_noise_std"], rng)
            pad_ctr += 1
        frames_rgb_t.append(frame_t)
    clip = torch.stack(frames_rgb_t, dim=0)                                       # (T, C, H, W) uint8
    pixel_values = pipeline["swin_eval_transform"](clip).unsqueeze(0).to(device)  # (1, C, T, H, W)

    logits = pipeline["swin_model"](pixel_values=pixel_values)
    probs = F.softmax(logits, dim=1)[0]
    pred_id = int(torch.argmax(probs).item())

    return {
        "prediction": ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "prob_real": float(probs[0]),
        "prob_fake": float(probs[1]),
    }


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--video", required=True)
    parser.add_argument("--checkpoint-dir", default="/kaggle/working/outputs")
    args = parser.parse_args()

    print(f"Loading pipeline from {args.checkpoint_dir} ...")
    pipeline = load_pipeline(args.checkpoint_dir)
    print(f"Running inference on {args.video} ...")
    result = predict_video(args.video, pipeline)
    print(json.dumps(result, indent=2))
